# 🎙️ Speech & Audio Processing - Whisper on Colab

**Transcribe audio and process speech with 100% privacy**

![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)

---

## Features

- \ud83c\udfa4 **Speech-to-Text**: Transcribe audio with Whisper
- \ud83c\udfa4 **Multi-language**: Support for 100+ languages
- \ud83c\udfa4 **Audio Classification**: Identify sounds
- \ud83c\udfa4 **100% Private**: All processing in Colab

---


In [ ]:
# Configuration
MODEL_SIZE = "base"  # @param ["tiny", "base", "small", "medium", "large"]

print(f"\ud83d\udc49 Model: Whisper {MODEL_SIZE}")

## Step 1: Setup

In [ ]:
import os

# Security
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = "/content/.hf_cache"
!mkdir -p /content/.hf_cache

# Install
!pip install -q transformers datasets librosa scipy soundfile

print("\u2705 Setup complete!")

## Step 2: Load Whisper Model

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch

model_name = f"openai/whisper-{MODEL_SIZE}"
print(f"Loading {model_name}...")

processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"\u2705 Model loaded on {device}!")

## Step 3: Transcribe Function

In [ ]:
def transcribe(audio_path=None, audio_array=None, sampling_rate=16000, language="en"):
    """
    Transcribe audio to text.
    
    Args:
        audio_path: Path to audio file OR
        audio_array: numpy array of audio samples
        sampling_rate: Audio sampling rate (default 16000)
        language: Language code (e.g., "en", "es", "fr", None for auto)
    """
    # Load audio
    if audio_path:
        import librosa
        audio, sr = librosa.load(audio_path, sr=sampling_rate)
    else:
        audio = audio_array
        sr = sampling_rate
    
    # Process
    input_features = processor(audio, sampling_rate=sr, return_tensors="pt").input_features
    input_features = input_features.to(device)
    
    # Generate
    forced_decoder_ids = processor.get_decoder_prompt_ids(language=language)
    
    with torch.no_grad():
        predicted_ids = model.generate(input_features, forced_decoder_ids=forced_decoder_ids)
    
    # Decode
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    
    return transcription

print("\u2705 Transcription function ready!")

## Step 4: Create Test Audio (or use your own)

In [ ]:
# Generate a simple test audio (you can replace this with your own audio)
import numpy as np
from scipy.io.wavfile import write

# Parameters
sample_rate = 16000
duration = 5  # seconds

# Generate a simple tone (replace this with your audio file)
frequency = 440  # Hz
t = np.linspace(0, duration, int(sample_rate * duration))
audio = 0.3 * np.sin(2 * np.pi * frequency * t)

# Save test audio
test_audio_path = "/content/test_audio.wav"
write(test_audio_path, sample_rate, (audio * 32767).astype(np.int16))

print(f"\ud83c\udfa7 Test audio created: {test_audio_path}")
print("   (Replace with your own audio file for real transcription)")

## Step 5: Transcribe!

In [ ]:
# Transcribe audio file
print("\ud83c\udfa4 Transcribing test audio...")
print("(This is a tone, so transcription will be empty. Use your own audio!)\n")

transcription = transcribe(audio_path=test_audio_path, language="en")

print(f"\ud83d\udcce Transcription:\n{transcription if transcription else '[No speech detected]'}")

## Step 6: Upload Your Own Audio

In [ ]:
# Upload audio file
from google.colab import files

print("\ud83d\udce4 Upload your audio file:")
uploaded = files.upload()

if uploaded:
    # Get the filename
    audio_filename = list(uploaded.keys())[0]
    print(f"\u2705 Uploaded: {audio_filename}")
    
    # Transcribe
    print("\ud83c\udfa4 Transcribing...")
    transcription = transcribe(audio_path=audio_filename, language="en")
    
    print(f"\n\ud83d\udcce Transcription:\n{transcription}")

## Step 7: Batch Processing

In [ ]:
# Process multiple audio files
def batch_transcribe(audio_paths, language="en"):
    """Transcribe multiple audio files."""
    results = {}
    
    for path in audio_paths:
        print(f"Transcribing: {path}")
        try:
            results[path] = transcribe(audio_path=path, language=language)
        except Exception as e:
            results[path] = f"Error: {e}"
    
    return results

print("\u2705 Batch transcription ready!")

## Step 8: Memory Cleanup

In [ ]:
import gc

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("\u2705 Memory cleared!")

cleanup()

---

## Whisper Model Sizes

| Size | Params | English | Multi-lang | Speed |
|------|--------|---------|------------|-------|
| tiny | 39M | \u2b50\u2b50 | \u2b50 | \u2b50\u2b50\u2b50\u2b50\u2b50 |
| base | 74M | \u2b50\u2b50\u2b50 | \u2b50\u2b50 | \u2b50\u2b50\u2b50\u2b50 |
| small | 244M | \u2b50\u2b50\u2b50\u2b50 | \u2b50\u2b50\u2b50 | \u2b50\u2b50\u2b50 |
| medium | 769M | \u2b50\u2b50\u2b50\u2b50\u2b50 | \u2b50\u2b50\u2b50\u2b50 | \u2b50\u2b50 |

## Supported Languages

English, Spanish, French, German, Italian, Portuguese, Russian, Chinese, Japanese, Korean, Arabic, Hindi, and 100+ more!

Set `language=None` for automatic language detection.

---

**Model**: [OpenAI Whisper on HuggingFace](https://huggingface.co/openai/whisper-base)

**Star**: [GitHub](https://github.com/unn-Known1/huggingface-colab-secure-setup)